# Random Forest

## Random Forest classification and evaluation

Trains and optimizes a multi-output Random Forest for spectral type and peculiar class classification, evaluates performance on validation and test sets, generates confusion matrices, and analyzes feature importance.

In [ ]:
from pathlib import Path
import re, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix

# ============================================================
# CONFIGURATION
# ============================================================
FOLDER=Path("RF Results"); FOLDER.mkdir(exist_ok=True)
FILES={"Train":"train.csv","Validation":"val.csv","Test":"test.csv"}
TARGETS=["spectral_type_int","dwarf_type"]; MAGS=["Hmag","Jmag"]
FIELDREF=["fieldref_A","fieldref_YJ","fieldref_H","fieldref_K","fieldref_D"]
SPT_LABELS=list(range(30)); DWARF_LABELS=[-3,-2,-1,0,1,2]
DWARF_NAMES={-3:"esd",-2:"sd",-1:"d/sd",0:"field",1:"gamma",2:"beta"}
BIN_RE=re.compile(r"^\s*\(?\s*\d+(?:\.\d+)?\s*,\s*\d+(?:\.\d+)?\s*\)?\s*$")
N_TRIALS,SEED=30,42

plt.rcParams.update({"font.family":"serif","font.serif":["Times New Roman","Times","DejaVu Serif"],
                     "mathtext.fontset":"stix","font.size":13})

def spt_name(x):
    x=int(x); return f"M{x}" if x<10 else f"L{x-10}" if x<20 else f"T{x-20}" if x<30 else str(x)

def load_csv(file):
    d=pd.read_csv(file,sep=None,engine="python"); d.columns=d.columns.astype(str).str.strip(); return d

data={name:load_csv(file) for name,file in FILES.items()}

# ============================================================
# FEATURES
# ============================================================
for name,d in data.items():
    missing=[c for c in TARGETS+MAGS+FIELDREF if c not in d]
    if missing: raise ValueError(f"{name} is missing: {missing}")

common=[c for c in data["Train"] if all(c in d for d in data.values())]
slope_cols=[c for c in common if c.startswith("slope_")]
bin_cols=[c for c in common if BIN_RE.fullmatch(c)]
feature_cols=list(dict.fromkeys(MAGS+slope_cols+bin_cols+FIELDREF))
feature_cols=[c for c in feature_cols if pd.to_numeric(data["Train"][c],errors="coerce").notna().any()]

if not slope_cols or not bin_cols: raise ValueError("No slopes or spectral bins were detected.")

print(f"Magnitudes: {MAGS}\nSlopes: {len(slope_cols)}\nSpectral bins: {len(bin_cols)}")
print(f"Field-reference features: {FIELDREF}\nTotal features: {len(feature_cols)}")
if len(slope_cols)!=16: print(f"WARNING: {len(slope_cols)} slopes detected; 16 expected.")
if len(bin_cols)!=67: print(f"WARNING: {len(bin_cols)} bins detected; 67 expected.")

def prepare(d):
    X=d[feature_cols].apply(pd.to_numeric,errors="coerce")
    y=d[TARGETS].apply(pd.to_numeric,errors="coerce")
    if y.isna().any().any(): raise ValueError("Missing or nonnumeric target values found.")
    return X,y.astype(int)

X_train,y_train=prepare(data["Train"])
X_val,y_val=prepare(data["Validation"])
X_test,y_test=prepare(data["Test"])
print(f"\nTrain: {X_train.shape} | Validation: {X_val.shape} | Test: {X_test.shape}")

# ============================================================
# HYPERPARAMETER SEARCH
# Priority: Dwarf BAcc > Dwarf Acc > SpT BAcc > SpT Acc
# ============================================================
ZHOU={"n_estimators":700,"max_depth":None,"max_features":"sqrt",
      "min_samples_split":5,"min_samples_leaf":1}

SPACE={"n_estimators":[300,500,700,1000],
       "max_depth":[None,20,40,60],
       "max_features":["sqrt","log2",0.25,0.50],
       "min_samples_split":[2,5,10],
       "min_samples_leaf":[1,2,4]}

imputer=SimpleImputer(strategy="median")
Xt,Xv=imputer.fit_transform(X_train),imputer.transform(X_val)
yt,yv=y_train.to_numpy(),y_val.to_numpy()

candidates=[("Zhou",ZHOU)]+[("Random",p) for p in ParameterSampler(SPACE,n_iter=N_TRIALS,random_state=SEED)]
results,seen,best_rank,best_params,best_metrics=[],set(),(-np.inf,)*4,None,None

for i,(source,p) in enumerate(candidates,1):
    key=tuple(sorted(p.items()))
    if key in seen: continue
    seen.add(key)

    clf=MultiOutputClassifier(RandomForestClassifier(**p,random_state=SEED,n_jobs=-1))
    clf.fit(Xt,yt); pred=clf.predict(Xv)

    spt_acc=accuracy_score(yv[:,0],pred[:,0])
    spt_bacc=balanced_accuracy_score(yv[:,0],pred[:,0])
    dwarf_acc=accuracy_score(yv[:,1],pred[:,1])
    dwarf_bacc=balanced_accuracy_score(yv[:,1],pred[:,1])
    mean_acc=(spt_acc+dwarf_acc)/2
    mean_bacc=(spt_bacc+dwarf_bacc)/2

    row={"source":source,**p,"SpT_Acc":spt_acc,"SpT_BAcc":spt_bacc,
         "Dwarf_Acc":dwarf_acc,"Dwarf_BAcc":dwarf_bacc,
         "Mean_Acc":mean_acc,"Mean_BAcc":mean_bacc}
    results.append(row)

    rank=(dwarf_bacc,dwarf_acc,spt_bacc,spt_acc)
    if rank>best_rank:
        best_rank,best_params,best_metrics=rank,p.copy(),row.copy()

    print(f"{i:02d} | {source:6s} | SpT Acc={spt_acc:.4f} BAcc={spt_bacc:.4f} | "
          f"Dwarf Acc={dwarf_acc:.4f} BAcc={dwarf_bacc:.4f}")

tuning_df=(pd.DataFrame(results)
           .sort_values(["Dwarf_BAcc","Dwarf_Acc","SpT_BAcc","SpT_Acc"],ascending=False)
           .reset_index(drop=True))

print("\nBEST HYPERPARAMETERS — DWARF TYPE PRIORITIZED")
for k,v in best_params.items(): print(f"{k}: {v}")
print(f"Validation dwarf accuracy: {best_metrics['Dwarf_Acc']:.4f}")
print(f"Validation dwarf balanced accuracy: {best_metrics['Dwarf_BAcc']:.4f}")
print(f"Validation spectral-type accuracy: {best_metrics['SpT_Acc']:.4f}")
print(f"Validation spectral-type balanced accuracy: {best_metrics['SpT_BAcc']:.4f}")

print("\nTOP 10 CONFIGURATIONS")
print(tuning_df.head(10).to_string(index=False))

tuning_df.to_csv(FOLDER/"rf_hyperparameter_tuning.csv",index=False)
pd.DataFrame([{**best_params,
               "validation_dwarf_accuracy":best_metrics["Dwarf_Acc"],
               "validation_dwarf_balanced_accuracy":best_metrics["Dwarf_BAcc"],
               "validation_spt_accuracy":best_metrics["SpT_Acc"],
               "validation_spt_balanced_accuracy":best_metrics["SpT_BAcc"]}]
             ).to_csv(FOLDER/"rf_best_hyperparameters.csv",index=False)

# ============================================================
# FINAL MODEL
# ============================================================
model=Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("rf",MultiOutputClassifier(RandomForestClassifier(**best_params,random_state=SEED,n_jobs=-1)))
])
model.fit(X_train,y_train)
print("\nFinal Random Forest trained.")

# ============================================================
# CONFUSION MATRICES
# ============================================================
def plot_cm(y_true,y_pred,labels,names,file,kind):
    cm=confusion_matrix(y_true,y_pred,labels=labels,normalize="true")

    if kind=="spt":
        fig,ax=plt.subplots(figsize=(15,13),dpi=100)
        tick_fs,label_fs,text_fs,cbar_fs,cbar_tick=22,29,15,27,22
        rotation,shrink=90,.75
    else:
        fig,ax=plt.subplots(figsize=(10,8))
        tick_fs,label_fs,text_fs,cbar_fs,cbar_tick=20,24,24,20,14
        rotation,shrink=45,1

    im=ax.imshow(cm,cmap="Blues",vmin=0,vmax=1,aspect="auto")
    ax.set_xticks(range(len(names)),names,rotation=rotation)
    ax.set_yticks(range(len(names)),names)
    ax.tick_params(axis="both",labelsize=tick_fs)
    ax.set_xlabel("Predicted label",fontsize=label_fs,labelpad=16)
    ax.set_ylabel("Adopted label",fontsize=label_fs,labelpad=16)

    for i,j in np.ndindex(cm.shape):
        v=cm[i,j]
        text="0" if v==0 else "<1" if v*100<1 else f"{v*100:.0f}"
        ax.text(j,i,text,ha="center",va="center",fontsize=text_fs,
                color="white" if v>=.55 else "black")

    cbar=fig.colorbar(im,ax=ax,pad=.02 if kind=="spt" else .03,
                      shrink=shrink,fraction=.035,aspect=30)
    cbar.set_label("Normalized frequency",fontsize=cbar_fs,labelpad=12)
    cbar.ax.tick_params(labelsize=cbar_tick)
    plt.tight_layout()
    fig.savefig(file,dpi=600,bbox_inches="tight",facecolor="white")
    plt.show()
    return cm

# ============================================================
# EVALUATION
# ============================================================
def evaluate(name,d,X,y):
    true=y.to_numpy(int); pred=model.predict(X).astype(int)
    ts,td=true[:,0],true[:,1]; ps,pd_=pred[:,0],pred[:,1]
    err=np.abs(ts-ps)

    summary={
        "Dataset":name,"Samples":len(y),"Features":X.shape[1],
        "Exact joint":np.mean((ts==ps)&(td==pd_)),
        "Joint ±1 SpT":np.mean((err<=1)&(td==pd_)),
        "Joint ±3 SpT":np.mean((err<=3)&(td==pd_)),
        "SpT accuracy":accuracy_score(ts,ps),
        "SpT balanced accuracy":balanced_accuracy_score(ts,ps),
        "SpT ±1":np.mean(err<=1),"SpT ±3":np.mean(err<=3),
        "Dwarf accuracy":accuracy_score(td,pd_),
        "Dwarf balanced accuracy":balanced_accuracy_score(td,pd_)
    }

    print("\n"+"="*72+f"\n{name}\n"+"="*72)
    for k,v in summary.items():
        if k not in ["Dataset","Samples","Features"]: print(f"{k:27s}: {v:.4f}")

    print("\nSPECTRAL-TYPE REPORT")
    print(classification_report(ts,ps,labels=SPT_LABELS,
          target_names=[spt_name(x) for x in SPT_LABELS],zero_division=0))

    print("\nDWARF-TYPE REPORT")
    print(classification_report(td,pd_,labels=DWARF_LABELS,
          target_names=[DWARF_NAMES[x] for x in DWARF_LABELS],zero_division=0))

    tag=name.lower()
    cm_spt=plot_cm(ts,ps,SPT_LABELS,[spt_name(x) for x in SPT_LABELS],
                   FOLDER/f"cm_rf_spt_{tag}.png","spt")
    cm_dwarf=plot_cm(td,pd_,DWARF_LABELS,[DWARF_NAMES[x] for x in DWARF_LABELS],
                     FOLDER/f"cm_rf_dwarf_{tag}.png","dwarf")

    ids=[c for c in ["Parent","Parent_Spectral_Type","Names","Type_Origen","Hmag","Jmag"] if c in d]
    out=d[ids].reset_index(drop=True).copy()
    out["true_spectral_type_int"],out["pred_spectral_type_int"]=ts,ps
    out["true_spectral_type"],out["pred_spectral_type"]=[spt_name(x) for x in ts],[spt_name(x) for x in ps]
    out["true_dwarf_type_int"],out["pred_dwarf_type_int"]=td,pd_
    out["true_dwarf_type"],out["pred_dwarf_type"]=[DWARF_NAMES.get(x,str(x)) for x in td],[DWARF_NAMES.get(x,str(x)) for x in pd_]
    out["spectral_type_error"],out["abs_spectral_type_error"]=ps-ts,err
    out["correct_spectral_type"]=ts==ps
    out["correct_spectral_type_pm1"]=err<=1
    out["correct_spectral_type_pm3"]=err<=3
    out["correct_dwarf_type"]=td==pd_
    out["correct_exact_match"]=(ts==ps)&(td==pd_)

    pd.DataFrame(cm_spt,index=[spt_name(x) for x in SPT_LABELS],
                 columns=[spt_name(x) for x in SPT_LABELS]).to_csv(FOLDER/f"rf_cm_spt_{tag}.csv")
    pd.DataFrame(cm_dwarf,index=[DWARF_NAMES[x] for x in DWARF_LABELS],
                 columns=[DWARF_NAMES[x] for x in DWARF_LABELS]).to_csv(FOLDER/f"rf_cm_dwarf_{tag}.csv")
    return out,summary

val_predictions,val_summary=evaluate("Validation",data["Validation"],X_val,y_val)
test_predictions,test_summary=evaluate("Test",data["Test"],X_test,y_test)
summary_df=pd.DataFrame([val_summary,test_summary])

print("\nSUMMARY")
print(summary_df.to_string(index=False))

# ============================================================
# FEATURE IMPORTANCE — DWARF TYPE PRIORITIZED
# ============================================================
rf_spt,rf_dwarf=model.named_steps["rf"].estimators_

importance_df=pd.DataFrame({
    "feature":feature_cols,
    "spectral_type_importance":rf_spt.feature_importances_,
    "dwarf_type_importance":rf_dwarf.feature_importances_
})
importance_df["mean_importance"]=importance_df[["spectral_type_importance","dwarf_type_importance"]].mean(axis=1)

def feature_group(c):
    if c in MAGS: return "Magnitude"
    if c.startswith("slope_"): return "Slope"
    if BIN_RE.fullmatch(c): return "Spectral bin"
    if c in FIELDREF: return "Field reference"
    return "Other"

importance_df["feature_group"]=importance_df["feature"].map(feature_group)
importance_df=importance_df.sort_values("dwarf_type_importance",ascending=False)

importance_by_group=(importance_df.groupby("feature_group",as_index=False)
    .agg(n_features=("feature","size"),
         spectral_type_importance=("spectral_type_importance","sum"),
         dwarf_type_importance=("dwarf_type_importance","sum"),
         mean_importance=("mean_importance","sum"))
    .sort_values("dwarf_type_importance",ascending=False))

print("\nTOP 30 FEATURES FOR DWARF-TYPE CLASSIFICATION")
print(importance_df.head(30).to_string(index=False))
print("\nIMPORTANCE BY GROUP")
print(importance_by_group.to_string(index=False))

top=importance_df.head(30).sort_values("dwarf_type_importance")
plt.figure(figsize=(10,8))
plt.barh(top["feature"],top["dwarf_type_importance"])
plt.xlabel("Random Forest importance for dwarf-type classification",fontsize=15)
plt.ylabel("Feature",fontsize=15)
plt.title("Top 30 features for dwarf-type classification",fontsize=16)
plt.tick_params(axis="both",labelsize=11)
plt.grid(axis="x",alpha=.3)
plt.tight_layout()
plt.savefig(FOLDER/"rf_top30_dwarf_feature_importance.png",dpi=600,bbox_inches="tight")
plt.show()

# ============================================================
# SAVE
# ============================================================
val_predictions.to_csv(FOLDER/"rf_validation_predictions.csv",index=False)
test_predictions.to_csv(FOLDER/"rf_test_predictions.csv",index=False)
summary_df.to_csv(FOLDER/"rf_summary.csv",index=False)
importance_df.to_csv(FOLDER/"rf_feature_importance.csv",index=False)
importance_by_group.to_csv(FOLDER/"rf_feature_importance_by_group.csv",index=False)

print(f"\nAll results saved in: {FOLDER.resolve()}")

## Random Forest classification with fixed hyperparameters

Trains a multi-output Random Forest using the Zhou et al. hyperparameters, evaluates spectral type and peculiar class performance, generates confusion matrices, and computes feature importance.

In [ ]:
from pathlib import Path
import re, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix

FOLDER=Path("RF Results"); FOLDER.mkdir(exist_ok=True)
FILES={"Train":"train.csv","Validation":"val.csv","Test":"test.csv"}
TARGETS=["spectral_type_int","dwarf_type"]; MAGS=["Hmag","Jmag"]
FIELDREF=["fieldref_A","fieldref_YJ","fieldref_H","fieldref_K","fieldref_D"]
SPT_LABELS=list(range(30)); DWARF_LABELS=[-3,-2,-1,0,1,2]
DWARF_NAMES={-3:"esd",-2:"sd",-1:"d/sd",0:"field",1:"gamma",2:"beta"}
BIN_RE=re.compile(r"^\s*\(?\s*\d+(?:\.\d+)?\s*,\s*\d+(?:\.\d+)?\s*\)?\s*$")
SEED=42
RF_PARAMS={"n_estimators":700,"max_depth":None,"max_features":"sqrt","min_samples_split":5,"min_samples_leaf":1}
plt.rcParams.update({"font.family":"serif","font.serif":["Times New Roman","Times","DejaVu Serif"],"mathtext.fontset":"stix","font.size":13})

def spt_name(x):
    x=int(x); return f"M{x}" if x<10 else f"L{x-10}" if x<20 else f"T{x-20}" if x<30 else str(x)

def load_csv(file):
    d=pd.read_csv(file,sep=None,engine="python"); d.columns=d.columns.astype(str).str.strip(); return d

data={name:load_csv(file) for name,file in FILES.items()}

for name,d in data.items():
    missing=[c for c in TARGETS+MAGS+FIELDREF if c not in d]
    if missing: raise ValueError(f"{name} is missing: {missing}")

common=[c for c in data["Train"] if all(c in d for d in data.values())]
slope_cols=[c for c in common if c.startswith("slope_")]
bin_cols=[c for c in common if BIN_RE.fullmatch(c)]
feature_cols=list(dict.fromkeys(MAGS+slope_cols+bin_cols+FIELDREF))
feature_cols=[c for c in feature_cols if pd.to_numeric(data["Train"][c],errors="coerce").notna().any()]
if not slope_cols or not bin_cols: raise ValueError("No slopes or spectral bins were detected.")
print(f"Magnitudes: {MAGS}\nSlopes: {len(slope_cols)}\nSpectral bins: {len(bin_cols)}")
print(f"Field-reference features: {FIELDREF}\nTotal features: {len(feature_cols)}")
if len(slope_cols)!=16: print(f"WARNING: {len(slope_cols)} slopes detected; 16 expected.")
if len(bin_cols)!=67: print(f"WARNING: {len(bin_cols)} bins detected; 67 expected.")

def prepare(d):
    X=d[feature_cols].apply(pd.to_numeric,errors="coerce")
    y=d[TARGETS].apply(pd.to_numeric,errors="coerce")
    if y.isna().any().any(): raise ValueError("Missing or nonnumeric target values found.")
    return X,y.astype(int)

X_train,y_train=prepare(data["Train"])
X_val,y_val=prepare(data["Validation"])
X_test,y_test=prepare(data["Test"])
print(f"\nTrain: {X_train.shape} | Validation: {X_val.shape} | Test: {X_test.shape}")
print("\nRandom Forest hyperparameters fixed following Zhou et al.")
for k,v in RF_PARAMS.items(): print(f"{k}: {v}")
pd.DataFrame([RF_PARAMS]).to_csv(FOLDER/"rf_fixed_hyperparameters_zhou.csv",index=False)
model=Pipeline([("imputer",SimpleImputer(strategy="median")),("rf",MultiOutputClassifier(RandomForestClassifier(**RF_PARAMS,random_state=SEED,n_jobs=-1)))])
model.fit(X_train,y_train)
print("\nFinal Random Forest trained.")

def plot_cm(y_true,y_pred,labels,names,file,kind):
    cm=confusion_matrix(y_true,y_pred,labels=labels,normalize="true")
    if kind=="spt":
        fig,ax=plt.subplots(figsize=(15,13),dpi=100)
        tick_fs,label_fs,text_fs,cbar_fs,cbar_tick=22,29,15,27,22
        rotation,shrink=90,.75
    else:
        fig,ax=plt.subplots(figsize=(10,8))
        tick_fs,label_fs,text_fs,cbar_fs,cbar_tick=20,24,24,20,14
        rotation,shrink=45,1
    im=ax.imshow(cm,cmap="Blues",vmin=0,vmax=1,aspect="auto")
    ax.set_xticks(range(len(names)),names,rotation=rotation)
    ax.set_yticks(range(len(names)),names)
    ax.tick_params(axis="both",labelsize=tick_fs)
    ax.set_xlabel("Predicted label",fontsize=label_fs,labelpad=16)
    ax.set_ylabel("Adopted label",fontsize=label_fs,labelpad=16)
    for i,j in np.ndindex(cm.shape):
        v=cm[i,j]; text = "0" if v==0 else "<0.1" if v<0.1 else f"{v:.2f}"
        ax.text(j,i,text,ha="center",va="center",fontsize=text_fs,color="white" if v>=.55 else "black")
    cbar=fig.colorbar(im,ax=ax,pad=.02 if kind=="spt" else .03,shrink=shrink,fraction=.035,aspect=30)
    cbar.set_label("Normalized frequency",fontsize=cbar_fs,labelpad=12)
    cbar.ax.tick_params(labelsize=cbar_tick)
    plt.tight_layout()
    fig.savefig(file,dpi=600,bbox_inches="tight",facecolor="white")
    plt.show()
    return cm

def evaluate(name,d,X,y):
    true=y.to_numpy(int); pred=model.predict(X).astype(int)
    ts,td=true[:,0],true[:,1]; ps,pd_=pred[:,0],pred[:,1]; err=np.abs(ts-ps)
    summary={
        "Dataset":name,"Samples":len(y),"Features":X.shape[1],
        "Exact joint":np.mean((ts==ps)&(td==pd_)),
        "Joint ±1 SpT":np.mean((err<=1)&(td==pd_)),
        "Joint ±3 SpT":np.mean((err<=3)&(td==pd_)),
        "SpT accuracy":accuracy_score(ts,ps),
        "SpT balanced accuracy":balanced_accuracy_score(ts,ps),
        "SpT ±1":np.mean(err<=1),"SpT ±3":np.mean(err<=3),
        "Dwarf accuracy":accuracy_score(td,pd_),
        "Dwarf balanced accuracy":balanced_accuracy_score(td,pd_)}
    print("\n"+"="*72+f"\n{name}\n"+"="*72)
    for k,v in summary.items():
        if k not in ["Dataset","Samples","Features"]: print(f"{k:27s}: {v:.4f}")
    print("\nSPECTRAL-TYPE REPORT")
    print(classification_report(ts,ps,labels=SPT_LABELS,target_names=[spt_name(x) for x in SPT_LABELS],zero_division=0))
    print("\nDWARF-TYPE REPORT")
    print(classification_report(td,pd_,labels=DWARF_LABELS,target_names=[DWARF_NAMES[x] for x in DWARF_LABELS],zero_division=0))
    tag=name.lower()
    cm_spt=plot_cm(ts,ps,SPT_LABELS,[spt_name(x) for x in SPT_LABELS],FOLDER/f"cm_rf_spt_{tag}.png","spt")
    cm_dwarf=plot_cm(td,pd_,DWARF_LABELS,[DWARF_NAMES[x] for x in DWARF_LABELS],FOLDER/f"cm_rf_dwarf_{tag}.png","dwarf")
    ids=[c for c in ["Parent","Parent_Spectral_Type","Names","Type_Origen","Hmag","Jmag"] if c in d]
    out=d[ids].reset_index(drop=True).copy()
    out["true_spectral_type_int"],out["pred_spectral_type_int"]=ts,ps
    out["true_spectral_type"],out["pred_spectral_type"]=[spt_name(x) for x in ts],[spt_name(x) for x in ps]
    out["true_dwarf_type_int"],out["pred_dwarf_type_int"]=td,pd_
    out["true_dwarf_type"],out["pred_dwarf_type"]=[DWARF_NAMES.get(x,str(x)) for x in td],[DWARF_NAMES.get(x,str(x)) for x in pd_]
    out["spectral_type_error"],out["abs_spectral_type_error"]=ps-ts,err
    out["correct_spectral_type"],out["correct_spectral_type_pm1"],out["correct_spectral_type_pm3"]=ts==ps,err<=1,err<=3
    out["correct_dwarf_type"],out["correct_exact_match"]=td==pd_,(ts==ps)&(td==pd_)
    pd.DataFrame(cm_spt,index=[spt_name(x) for x in SPT_LABELS],columns=[spt_name(x) for x in SPT_LABELS]).to_csv(FOLDER/f"rf_cm_spt_{tag}.csv")
    pd.DataFrame(cm_dwarf,index=[DWARF_NAMES[x] for x in DWARF_LABELS],columns=[DWARF_NAMES[x] for x in DWARF_LABELS]).to_csv(FOLDER/f"rf_cm_dwarf_{tag}.csv")
    return out,summary

val_predictions,val_summary=evaluate("Validation",data["Validation"],X_val,y_val)
test_predictions,test_summary=evaluate("Test",data["Test"],X_test,y_test)
summary_df=pd.DataFrame([val_summary,test_summary])

print("\nSUMMARY")
print(summary_df.to_string(index=False))

rf_spt,rf_dwarf=model.named_steps["rf"].estimators_
importance_df=pd.DataFrame({"feature":feature_cols,"spectral_type_importance":rf_spt.feature_importances_,"dwarf_type_importance":rf_dwarf.feature_importances_})
importance_df["mean_importance"]=importance_df[["spectral_type_importance","dwarf_type_importance"]].mean(axis=1)

def feature_group(c):
    if c in MAGS: return "Magnitude"
    if c.startswith("slope_"): return "Slope"
    if BIN_RE.fullmatch(c): return "Spectral bin"
    if c in FIELDREF: return "Field reference"
    return "Other"

importance_df["feature_group"]=importance_df["feature"].map(feature_group)
importance_df=importance_df.sort_values("dwarf_type_importance",ascending=False)
importance_by_group=(importance_df.groupby("feature_group",as_index=False)
    .agg(n_features=("feature","size"),spectral_type_importance=("spectral_type_importance","sum"),
         dwarf_type_importance=("dwarf_type_importance","sum"),mean_importance=("mean_importance","sum"))
    .sort_values("dwarf_type_importance",ascending=False))

print("\nTOP 30 FEATURES FOR DWARF-TYPE CLASSIFICATION")
print(importance_df.head(30).to_string(index=False))
print("\nIMPORTANCE BY GROUP")
print(importance_by_group.to_string(index=False))
top=importance_df.head(30).sort_values("dwarf_type_importance")
plt.figure(figsize=(10,8))
plt.barh(top["feature"],top["dwarf_type_importance"])
plt.xlabel("Random Forest importance for dwarf-type classification",fontsize=15)
plt.ylabel("Feature",fontsize=15)
plt.title("Top 30 features for dwarf-type classification",fontsize=16)
plt.tick_params(axis="both",labelsize=11)
plt.grid(axis="x",alpha=.3)
plt.tight_layout()
plt.savefig(FOLDER/"rf_top30_dwarf_feature_importance.png",dpi=600,bbox_inches="tight")
plt.show()
val_predictions.to_csv(FOLDER/"rf_validation_predictions.csv",index=False)
test_predictions.to_csv(FOLDER/"rf_test_predictions.csv",index=False)
summary_df.to_csv(FOLDER/"rf_summary.csv",index=False)
importance_df.to_csv(FOLDER/"rf_feature_importance.csv",index=False)
importance_by_group.to_csv(FOLDER/"rf_feature_importance_by_group.csv",index=False)
print(f"\nAll results saved in: {FOLDER.resolve()}")

# Support Vector Machines

## Support Vector Machine classification and evaluation

Trains and optimizes a multi-output Support Vector Machine for spectral type and peculiar class classification, evaluates performance on validation and test sets, and generates confusion matrices.

In [ ]:
from pathlib import Path
import re, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix

# ============================================================
# CONFIGURATION
# ============================================================
FOLDER=Path("SVM Results - 30"); FOLDER.mkdir(exist_ok=True)
FILES={"Train":"train.csv","Validation":"val.csv","Test":"test.csv"}
TARGETS=["spectral_type_int","dwarf_type"]; MAGS=["Jmag","Hmag"]
FIELDREF=["fieldref_A","fieldref_YJ","fieldref_H","fieldref_K","fieldref_D"]
SPT_LABELS=list(range(30)); DWARF_LABELS=[-3,-2,-1,0,1,2]
DWARF_NAMES={-3:"esd",-2:"sd",-1:"d/sd",0:"field",1:"gamma",2:"beta"}
BIN_RE=re.compile(r"^\s*\(?\s*\d+(?:\.\d+)?\s*,\s*\d+(?:\.\d+)?\s*\)?\s*$")
N_TRIALS,SEED=30,42

ZHOU={"C":10,"kernel":"rbf","gamma":"scale","decision_function_shape":"ovo"}
SPACE={"C":[.1,.5,1,2,5,10,20,50,100],
       "gamma":["scale","auto",1e-4,5e-4,1e-3,5e-3,1e-2,5e-2,1e-1]}

plt.rcParams.update({"font.family":"serif","font.serif":["Times New Roman","Times","DejaVu Serif"],
                     "mathtext.fontset":"stix","font.size":13})

def spt_name(x):
    x=int(x); return f"M{x}" if x<10 else f"L{x-10}" if x<20 else f"T{x-20}" if x<30 else str(x)

def load_csv(file):
    d=pd.read_csv(file,sep=None,engine="python"); d.columns=d.columns.astype(str).str.strip(); return d

data={name:load_csv(file) for name,file in FILES.items()}

# ============================================================
# FEATURES
# ============================================================
for name,d in data.items():
    missing=[c for c in TARGETS+MAGS+FIELDREF if c not in d]
    if missing: raise ValueError(f"{name} is missing: {missing}")

common=[c for c in data["Train"] if all(c in d for d in data.values())]
slope_cols=[c for c in common if c.startswith("slope_")]
bin_cols=[c for c in common if BIN_RE.fullmatch(c)]
feature_cols=list(dict.fromkeys(MAGS+slope_cols+bin_cols+FIELDREF))
feature_cols=[c for c in feature_cols if pd.to_numeric(data["Train"][c],errors="coerce").notna().any()]

if not slope_cols or not bin_cols: raise ValueError("No slopes or spectral bins were detected.")

print(f"Magnitudes: {MAGS}\nSlopes: {len(slope_cols)}\nSpectral bins: {len(bin_cols)}")
print(f"Field-reference features: {FIELDREF}\nTotal features: {len(feature_cols)}")
if len(bin_cols)!=67: print(f"WARNING: {len(bin_cols)} bins detected; 67 expected.")

def prepare(d):
    X=d[feature_cols].apply(pd.to_numeric,errors="coerce")
    y=d[TARGETS].apply(pd.to_numeric,errors="coerce")
    if y.isna().any().any(): raise ValueError("Missing or nonnumeric target values found.")
    return X,y.astype(int)

X_train,y_train=prepare(data["Train"])
X_val,y_val=prepare(data["Validation"])
X_test,y_test=prepare(data["Test"])
print(f"\nTrain: {X_train.shape} | Validation: {X_val.shape} | Test: {X_test.shape}")

# ============================================================
# TRAIN-ONLY PREPROCESSING
# ============================================================
imputer,scaler=SimpleImputer(strategy="median"),StandardScaler()
Xt=imputer.fit_transform(X_train); Xv=imputer.transform(X_val)
Xt=scaler.fit_transform(Xt); Xv=scaler.transform(Xv)
yt,yv=y_train.to_numpy(),y_val.to_numpy()

# Exactly 30 unique random configurations, excluding Zhou
grid=[{**p,"kernel":"rbf","decision_function_shape":"ovo"} for p in ParameterGrid(SPACE)]
grid=[p for p in grid if p!=ZHOU]
rng=np.random.default_rng(SEED)
random_params=[grid[i] for i in rng.choice(len(grid),N_TRIALS,replace=False)]
candidates=[("Zhou",ZHOU)]+[("Random",p) for p in random_params]

# ============================================================
# HYPERPARAMETER SEARCH
# Priority: Dwarf BAcc > Dwarf Acc > SpT BAcc > SpT Acc
# ============================================================
results,best_rank,best_params,best_metrics=[],(-np.inf,)*4,None,None

for i,(source,p) in enumerate(candidates,1):
    clf=MultiOutputClassifier(SVC(**p,cache_size=4096),n_jobs=2)
    clf.fit(Xt,yt); pred=clf.predict(Xv)

    spt_acc=accuracy_score(yv[:,0],pred[:,0])
    spt_bacc=balanced_accuracy_score(yv[:,0],pred[:,0])
    dwarf_acc=accuracy_score(yv[:,1],pred[:,1])
    dwarf_bacc=balanced_accuracy_score(yv[:,1],pred[:,1])

    row={"source":source,**p,"SpT_Acc":spt_acc,"SpT_BAcc":spt_bacc,
         "Dwarf_Acc":dwarf_acc,"Dwarf_BAcc":dwarf_bacc,
         "Mean_Acc":(spt_acc+dwarf_acc)/2,
         "Mean_BAcc":(spt_bacc+dwarf_bacc)/2}
    results.append(row)

    rank=(dwarf_bacc,dwarf_acc,spt_bacc,spt_acc)
    if rank>best_rank: best_rank,best_params,best_metrics=rank,p.copy(),row.copy()

    print(f"{i:02d} | {source:6s} | C={p['C']:>5} | gamma={str(p['gamma']):>7} | "
          f"SpT Acc={spt_acc:.4f} BAcc={spt_bacc:.4f} | "
          f"Dwarf Acc={dwarf_acc:.4f} BAcc={dwarf_bacc:.4f}")

tuning_df=(pd.DataFrame(results)
           .sort_values(["Dwarf_BAcc","Dwarf_Acc","SpT_BAcc","SpT_Acc"],ascending=False)
           .reset_index(drop=True))

print("\nBEST HYPERPARAMETERS — DWARF TYPE PRIORITIZED")
for k,v in best_params.items(): print(f"{k}: {v}")
print(f"Validation dwarf accuracy: {best_metrics['Dwarf_Acc']:.4f}")
print(f"Validation dwarf balanced accuracy: {best_metrics['Dwarf_BAcc']:.4f}")
print(f"Validation spectral-type accuracy: {best_metrics['SpT_Acc']:.4f}")
print(f"Validation spectral-type balanced accuracy: {best_metrics['SpT_BAcc']:.4f}")

print("\nTOP 10 CONFIGURATIONS")
print(tuning_df.head(10).to_string(index=False))

tuning_df.to_csv(FOLDER/"svm_hyperparameter_tuning.csv",index=False)
pd.DataFrame([{**best_params,
    "validation_dwarf_accuracy":best_metrics["Dwarf_Acc"],
    "validation_dwarf_balanced_accuracy":best_metrics["Dwarf_BAcc"],
    "validation_spt_accuracy":best_metrics["SpT_Acc"],
    "validation_spt_balanced_accuracy":best_metrics["SpT_BAcc"]}]
).to_csv(FOLDER/"svm_best_hyperparameters.csv",index=False)

# ============================================================
# FINAL MODEL
# ============================================================
model=Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler()),
    ("svm",MultiOutputClassifier(SVC(**best_params,cache_size=4096),n_jobs=2))
])
model.fit(X_train,y_train)
print("\nFinal SVM trained.")

# ============================================================
# CONFUSION MATRICES: VALUES FROM 0 TO 1
# ============================================================
def plot_cm(y_true,y_pred,labels,names,file,kind):
    cm=confusion_matrix(y_true,y_pred,labels=labels,normalize="true")
    spt=kind=="spt"
    fig,ax=plt.subplots(figsize=(15,13) if spt else (10,8),dpi=100)
    im=ax.imshow(cm,cmap="Blues",vmin=0,vmax=1,aspect="auto")
    ax.set_xticks(range(len(names)),names,rotation=90 if spt else 45)
    ax.set_yticks(range(len(names)),names)
    ax.tick_params(axis="both",labelsize=22 if spt else 20)
    ax.set_xlabel("Predicted label",fontsize=29 if spt else 24,labelpad=16)
    ax.set_ylabel("Adopted label",fontsize=29 if spt else 24,labelpad=16)

    for i,j in np.ndindex(cm.shape):
        v=cm[i,j]; text="0" if v==0 else "<0.01" if v<.01 else f"{v:.2f}"
        ax.text(j,i,text,ha="center",va="center",fontsize=15 if spt else 24,
                color="white" if v>=.55 else "black")

    cbar=fig.colorbar(im,ax=ax,pad=.02 if spt else .03,
                      shrink=.75 if spt else 1,fraction=.035,aspect=30)
    cbar.set_label("Normalized frequency",fontsize=27 if spt else 20,labelpad=12)
    cbar.ax.tick_params(labelsize=22 if spt else 14)
    plt.tight_layout()
    fig.savefig(file,dpi=600,bbox_inches="tight",facecolor="white")
    plt.show()
    return cm

# ============================================================
# EVALUATION
# ============================================================
def evaluate(name,d,X,y):
    true=y.to_numpy(int); pred=model.predict(X).astype(int)
    ts,td=true[:,0],true[:,1]; ps,pd_=pred[:,0],pred[:,1]; err=np.abs(ts-ps)

    summary={
        "Dataset":name,"Samples":len(y),"Features":X.shape[1],
        "Exact joint":np.mean((ts==ps)&(td==pd_)),
        "Joint ±1 SpT":np.mean((err<=1)&(td==pd_)),
        "Joint ±3 SpT":np.mean((err<=3)&(td==pd_)),
        "SpT accuracy":accuracy_score(ts,ps),
        "SpT balanced accuracy":balanced_accuracy_score(ts,ps),
        "SpT ±1":np.mean(err<=1),"SpT ±3":np.mean(err<=3),
        "Dwarf accuracy":accuracy_score(td,pd_),
        "Dwarf balanced accuracy":balanced_accuracy_score(td,pd_)
    }

    print("\n"+"="*72+f"\n{name}\n"+"="*72)
    for k,v in summary.items():
        if k not in ["Dataset","Samples","Features"]: print(f"{k:27s}: {v:.4f}")

    print("\nSPECTRAL-TYPE REPORT")
    print(classification_report(ts,ps,labels=SPT_LABELS,
          target_names=[spt_name(x) for x in SPT_LABELS],zero_division=0))

    print("\nDWARF-TYPE REPORT")
    print(classification_report(td,pd_,labels=DWARF_LABELS,
          target_names=[DWARF_NAMES[x] for x in DWARF_LABELS],zero_division=0))

    tag=name.lower()
    cm_spt=plot_cm(ts,ps,SPT_LABELS,[spt_name(x) for x in SPT_LABELS],
                   FOLDER/f"cm_svm_spt_{tag}.png","spt")
    cm_dwarf=plot_cm(td,pd_,DWARF_LABELS,[DWARF_NAMES[x] for x in DWARF_LABELS],
                     FOLDER/f"cm_svm_dwarf_{tag}.png","dwarf")

    ids=[c for c in ["Parent","Parent_Spectral_Type","Names","Type_Origen","Jmag","Hmag"] if c in d]
    out=d[ids].reset_index(drop=True).copy()
    out["true_spectral_type_int"],out["pred_spectral_type_int"]=ts,ps
    out["true_spectral_type"],out["pred_spectral_type"]=[spt_name(x) for x in ts],[spt_name(x) for x in ps]
    out["true_dwarf_type_int"],out["pred_dwarf_type_int"]=td,pd_
    out["true_dwarf_type"],out["pred_dwarf_type"]=[DWARF_NAMES.get(x,str(x)) for x in td],[DWARF_NAMES.get(x,str(x)) for x in pd_]
    out["spectral_type_error"],out["abs_spectral_type_error"]=ps-ts,err
    out["correct_spectral_type"]=ts==ps
    out["correct_spectral_type_pm1"]=err<=1
    out["correct_spectral_type_pm3"]=err<=3
    out["correct_dwarf_type"]=td==pd_
    out["correct_exact_match"]=(ts==ps)&(td==pd_)

    pd.DataFrame(cm_spt,index=[spt_name(x) for x in SPT_LABELS],
                 columns=[spt_name(x) for x in SPT_LABELS]).to_csv(FOLDER/f"svm_cm_spt_{tag}.csv")
    pd.DataFrame(cm_dwarf,index=[DWARF_NAMES[x] for x in DWARF_LABELS],
                 columns=[DWARF_NAMES[x] for x in DWARF_LABELS]).to_csv(FOLDER/f"svm_cm_dwarf_{tag}.csv")
    return out,summary

val_predictions,val_summary=evaluate("Validation",data["Validation"],X_val,y_val)
test_predictions,test_summary=evaluate("Test",data["Test"],X_test,y_test)
summary_df=pd.DataFrame([val_summary,test_summary])

print("\nSUMMARY")
print(summary_df.to_string(index=False))

# ============================================================
# SAVE
# ============================================================
val_predictions.to_csv(FOLDER/"svm_validation_predictions.csv",index=False)
test_predictions.to_csv(FOLDER/"svm_test_predictions.csv",index=False)
summary_df.to_csv(FOLDER/"svm_summary.csv",index=False)
pd.DataFrame([ZHOU]).to_csv(FOLDER/"svm_reference_hyperparameters_zhou.csv",index=False)

print(f"\nAll results saved in: {FOLDER.resolve()}")

## Support Vector Machine classification with fixed hyperparameters

Trains a multi-output Support Vector Machine using fixed hyperparameters, evaluates spectral type and peculiar class performance, and generates confusion matrices.

In [ ]:
from pathlib import Path
import re, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix

FOLDER=Path("SVM Results"); FOLDER.mkdir(exist_ok=True)
FILES={"Train":"train.csv","Validation":"val.csv","Test":"test.csv"}
TARGETS=["spectral_type_int","dwarf_type"]; MAGS=["Hmag","Jmag"]
FIELDREF=["fieldref_A","fieldref_YJ","fieldref_H","fieldref_K","fieldref_D"]
SPT_LABELS=list(range(30)); DWARF_LABELS=[-3,-2,-1,0,1,2]
DWARF_NAMES={-3:"esd",-2:"sd",-1:"d/sd",0:"field",1:"gamma",2:"beta"}
BIN_RE=re.compile(r"^\s*\(?\s*\d+(?:\.\d+)?\s*,\s*\d+(?:\.\d+)?\s*\)?\s*$")

plt.rcParams.update({"font.family":"serif","font.serif":["Times New Roman","Times","DejaVu Serif"],
                     "mathtext.fontset":"stix","font.size":13})

def spt_name(x):
    x=int(x); return f"M{x}" if x<10 else f"L{x-10}" if x<20 else f"T{x-20}" if x<30 else str(x)

def load(file):
    d=pd.read_csv(file,sep=None,engine="python"); d.columns=d.columns.astype(str).str.strip(); return d

data={k:load(v) for k,v in FILES.items()}

for name,d in data.items():
    missing=[c for c in TARGETS+MAGS+FIELDREF if c not in d]
    if missing: raise ValueError(f"{name} is missing: {missing}")

common=[c for c in data["Train"] if all(c in d for d in data.values())]
slopes=[c for c in common if c.startswith("slope_")]
bins=[c for c in common if BIN_RE.fullmatch(c)]
features=list(dict.fromkeys(MAGS+slopes+bins+FIELDREF))
features=[c for c in features if pd.to_numeric(data["Train"][c],errors="coerce").notna().any()]
if not slopes or not bins: raise ValueError("No slopes or spectral bins were detected.")
print(f"Magnitudes: {MAGS}\nSlopes: {len(slopes)}\nSpectral bins: {len(bins)}")
print(f"Field-reference features: {FIELDREF}\nTotal features: {len(features)}")

def prepare(d):
    X=d[features].apply(pd.to_numeric,errors="coerce")
    y=d[TARGETS].apply(pd.to_numeric,errors="coerce")
    if y.isna().any().any(): raise ValueError("Missing or nonnumeric target values found.")
    return X,y.astype(int)

X_train,y_train=prepare(data["Train"])
X_val,y_val=prepare(data["Validation"])
X_test,y_test=prepare(data["Test"])
print(f"\nTrain: {X_train.shape} | Validation: {X_val.shape} | Test: {X_test.shape}")
# Exact Zhou et al. configuration
SVM_PARAMS={"C":10,"kernel":"rbf","gamma":"scale","decision_function_shape":"ovo"}

model=Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler()),
    ("svm",MultiOutputClassifier(SVC(**SVM_PARAMS,cache_size=4096),n_jobs=-1))])

model.fit(X_train,y_train)
print("\nSVM trained with Zhou et al. hyperparameters:")
for k,v in SVM_PARAMS.items(): print(f"{k}: {v}")

def plot_cm(true,pred,labels,names,file,kind):
    cm=confusion_matrix(true,pred,labels=labels,normalize="true")  # Range 0–1
    spt=kind=="spt"
    fig,ax=plt.subplots(figsize=(15,13) if spt else (10,8),dpi=100)
    im=ax.imshow(cm,cmap="Blues",vmin=0,vmax=1,aspect="auto")
    ax.set_xticks(range(len(names)),names,rotation=90 if spt else 45)
    ax.set_yticks(range(len(names)),names)
    ax.tick_params(axis="both",labelsize=22 if spt else 20)
    ax.set_xlabel("Predicted label",fontsize=29 if spt else 24,labelpad=16)
    ax.set_ylabel("Adopted label",fontsize=29 if spt else 24,labelpad=16)
    for i,j in np.ndindex(cm.shape):
        v=cm[i,j]; text="0" if v==0 else "<0.01" if v<.01 else f"{v:.2f}"
        ax.text(j,i,text,ha="center",va="center",fontsize=15 if spt else 24,
                color="white" if v>=.55 else "black")
    cbar=fig.colorbar(im,ax=ax,pad=.02 if spt else .03,shrink=.75 if spt else 1,
                      fraction=.035,aspect=30)
    cbar.set_label("Normalized frequency",fontsize=27 if spt else 20,labelpad=12)
    cbar.ax.tick_params(labelsize=22 if spt else 14)
    plt.tight_layout()
    fig.savefig(file,dpi=600,bbox_inches="tight",facecolor="white")
    plt.show()
    return cm

def evaluate(name,d,X,y):
    true=y.to_numpy(int); pred=model.predict(X).astype(int)
    ts,td=true[:,0],true[:,1]; ps,pd_=pred[:,0],pred[:,1]; err=np.abs(ts-ps)

    summary={
        "Dataset":name,"Samples":len(y),"Features":X.shape[1],
        "Exact joint":np.mean((ts==ps)&(td==pd_)),
        "Joint ±1 SpT":np.mean((err<=1)&(td==pd_)),
        "Joint ±3 SpT":np.mean((err<=3)&(td==pd_)),
        "SpT accuracy":accuracy_score(ts,ps),
        "SpT balanced accuracy":balanced_accuracy_score(ts,ps),
        "SpT ±1":np.mean(err<=1),"SpT ±3":np.mean(err<=3),
        "Dwarf accuracy":accuracy_score(td,pd_),
        "Dwarf balanced accuracy":balanced_accuracy_score(td,pd_)    }
    print("\n"+"="*72+f"\n{name}\n"+"="*72)
    for k,v in summary.items():
        if k not in ["Dataset","Samples","Features"]: print(f"{k:27s}: {v:.4f}")
    print("\nSPECTRAL-TYPE REPORT")
    print(classification_report(ts,ps,labels=SPT_LABELS,
          target_names=[spt_name(x) for x in SPT_LABELS],zero_division=0))
    print("\nDWARF-TYPE REPORT")
    print(classification_report(td,pd_,labels=DWARF_LABELS,
          target_names=[DWARF_NAMES[x] for x in DWARF_LABELS],zero_division=0))
    tag=name.lower()
    cm_spt=plot_cm(ts,ps,SPT_LABELS,[spt_name(x) for x in SPT_LABELS],
                   FOLDER/f"cm_svm_spt_{tag}.png","spt")
    cm_dwarf=plot_cm(td,pd_,DWARF_LABELS,[DWARF_NAMES[x] for x in DWARF_LABELS],
                     FOLDER/f"cm_svm_dwarf_{tag}.png","dwarf")

    ids=[c for c in ["Parent","Parent_Spectral_Type","Names","Type_Origen","Hmag","Jmag"] if c in d]
    out=d[ids].reset_index(drop=True).copy()
    out["true_spectral_type_int"],out["pred_spectral_type_int"]=ts,ps
    out["true_spectral_type"],out["pred_spectral_type"]=[spt_name(x) for x in ts],[spt_name(x) for x in ps]
    out["true_dwarf_type_int"],out["pred_dwarf_type_int"]=td,pd_
    out["true_dwarf_type"],out["pred_dwarf_type"]=[DWARF_NAMES.get(x,str(x)) for x in td],[DWARF_NAMES.get(x,str(x)) for x in pd_]
    out["spectral_type_error"],out["abs_spectral_type_error"]=ps-ts,err
    out["correct_spectral_type"],out["correct_spectral_type_pm1"],out["correct_spectral_type_pm3"]=ts==ps,err<=1,err<=3
    out["correct_dwarf_type"],out["correct_exact_match"]=td==pd_,(ts==ps)&(td==pd_)
    pd.DataFrame(cm_spt,index=[spt_name(x) for x in SPT_LABELS],
                 columns=[spt_name(x) for x in SPT_LABELS]).to_csv(FOLDER/f"svm_cm_spt_{tag}.csv")
    pd.DataFrame(cm_dwarf,index=[DWARF_NAMES[x] for x in DWARF_LABELS],
                 columns=[DWARF_NAMES[x] for x in DWARF_LABELS]).to_csv(FOLDER/f"svm_cm_dwarf_{tag}.csv")
    return out,summary

val_predictions,val_summary=evaluate("Validation",data["Validation"],X_val,y_val)
test_predictions,test_summary=evaluate("Test",data["Test"],X_test,y_test)
summary=pd.DataFrame([val_summary,test_summary])
print("\nSUMMARY")
print(summary.to_string(index=False))
val_predictions.to_csv(FOLDER/"svm_validation_predictions.csv",index=False)
test_predictions.to_csv(FOLDER/"svm_test_predictions.csv",index=False)
summary.to_csv(FOLDER/"svm_summary.csv",index=False)
pd.DataFrame([SVM_PARAMS]).to_csv(FOLDER/"svm_hyperparameters_zhou.csv",index=False)
print(f"\nAll SVM results saved in: {FOLDER.resolve()}")

# K Nearest Neighbors

## K-Nearest Neighbors classification and evaluation

Trains and optimizes a K-Nearest Neighbors classifier for spectral type and peculiar class classification, evaluates performance on validation and test sets, and generates confusion matrices.

In [ ]:
from pathlib import Path
import re, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix

FOLDER=Path("/content/KNN Results - 30"); FOLDER.mkdir(parents=True,exist_ok=True)
FILES={"Train":"/content/train.csv","Validation":"/content/val.csv","Test":"/content/test.csv"}
TARGETS=["spectral_type_int","dwarf_type"]; MAGS=["Jmag","Hmag"]
FIELDREF=["fieldref_A","fieldref_YJ","fieldref_H","fieldref_K","fieldref_D"]
SPT_LABELS=list(range(30)); DWARF_LABELS=[-3,-2,-1,0,1,2]
DWARF_NAMES={-3:"esd",-2:"sd",-1:"d/sd",0:"field",1:"gamma",2:"beta"}
BIN_RE=re.compile(r"^\s*\(?\s*\d+(?:\.\d+)?\s*,\s*\d+(?:\.\d+)?\s*\)?\s*$")
N_TRIALS,SEED=30,42

ZHOU={"n_neighbors":29,"weights":"uniform","metric":"minkowski","p":2}
SPACE={"n_neighbors":[3,5,7,9,11,15,21,29,35,45,55],
       "weights":["uniform","distance"],
       "metric":["minkowski"],
       "p":[1,2]}

plt.rcParams.update({"font.family":"serif","font.serif":["Times New Roman","Times","DejaVu Serif"],"mathtext.fontset":"stix","font.size":13})

def spt_name(x):
    x=int(x); return f"M{x}" if x<10 else f"L{x-10}" if x<20 else f"T{x-20}" if x<30 else str(x)

def load_csv(file):
    d=pd.read_csv(file,sep=None,engine="python"); d.columns=d.columns.astype(str).str.strip(); return d

data={name:load_csv(file) for name,file in FILES.items()}

for name,d in data.items():
    missing=[c for c in TARGETS+MAGS+FIELDREF if c not in d]
    if missing: raise ValueError(f"{name} is missing: {missing}")

common=[c for c in data["Train"] if all(c in d for d in data.values())]
slope_cols=[c for c in common if c.startswith("slope_")]
bin_cols=[c for c in common if BIN_RE.fullmatch(c)]
feature_cols=list(dict.fromkeys(MAGS+slope_cols+bin_cols+FIELDREF))
feature_cols=[c for c in feature_cols if pd.to_numeric(data["Train"][c],errors="coerce").notna().any()]
if not slope_cols or not bin_cols: raise ValueError("No slopes or spectral bins were detected.")

print(f"Magnitudes: {MAGS}\nSlopes: {len(slope_cols)}\nSpectral bins: {len(bin_cols)}")
print(f"Field-reference features: {FIELDREF}\nTotal features: {len(feature_cols)}")
if len(slope_cols)!=16: print(f"WARNING: {len(slope_cols)} slopes detected; 16 expected.")
if len(bin_cols)!=67: print(f"WARNING: {len(bin_cols)} bins detected; 67 expected.")

def prepare(d):
    X=d[feature_cols].apply(pd.to_numeric,errors="coerce")
    y=d[TARGETS].apply(pd.to_numeric,errors="coerce")
    if y.isna().any().any(): raise ValueError("Missing or nonnumeric target values found.")
    return X,y.astype(int)

X_train,y_train=prepare(data["Train"])
X_val,y_val=prepare(data["Validation"])
X_test,y_test=prepare(data["Test"])
print(f"\nTrain: {X_train.shape} | Validation: {X_val.shape} | Test: {X_test.shape}")

imputer,scaler=SimpleImputer(strategy="median"),StandardScaler()
Xt=imputer.fit_transform(X_train); Xv=imputer.transform(X_val)
Xt=scaler.fit_transform(Xt); Xv=scaler.transform(Xv)
yt,yv=y_train.to_numpy(),y_val.to_numpy()

grid=list(ParameterGrid(SPACE))
grid=[p for p in grid if p!=ZHOU]
rng=np.random.default_rng(SEED)
random_params=[grid[i] for i in rng.choice(len(grid),N_TRIALS,replace=False)]
candidates=[("Zhou",ZHOU)]+[("Random",p) for p in random_params]

results,best_rank,best_params,best_metrics=[],(-np.inf,)*4,None,None

for i,(source,p) in enumerate(candidates,1):
    clf=MultiOutputClassifier(KNeighborsClassifier(**p,n_jobs=-1),n_jobs=-1)
    clf.fit(Xt,yt); pred=clf.predict(Xv)
    spt_acc=accuracy_score(yv[:,0],pred[:,0])
    spt_bacc=balanced_accuracy_score(yv[:,0],pred[:,0])
    dwarf_acc=accuracy_score(yv[:,1],pred[:,1])
    dwarf_bacc=balanced_accuracy_score(yv[:,1],pred[:,1])
    row={"source":source,**p,"SpT_Acc":spt_acc,"SpT_BAcc":spt_bacc,"Dwarf_Acc":dwarf_acc,"Dwarf_BAcc":dwarf_bacc,"Mean_Acc":(spt_acc+dwarf_acc)/2,"Mean_BAcc":(spt_bacc+dwarf_bacc)/2}
    results.append(row)
    rank=(dwarf_bacc,dwarf_acc,spt_bacc,spt_acc)
    if rank>best_rank: best_rank,best_params,best_metrics=rank,p.copy(),row.copy()
    print(f"{i:02d} | {source:6s} | k={p['n_neighbors']:>2} | weights={p['weights']:>8} | p={p['p']} | SpT Acc={spt_acc:.4f} BAcc={spt_bacc:.4f} | Dwarf Acc={dwarf_acc:.4f} BAcc={dwarf_bacc:.4f}")

tuning_df=(pd.DataFrame(results).sort_values(["Dwarf_BAcc","Dwarf_Acc","SpT_BAcc","SpT_Acc"],ascending=False).reset_index(drop=True))

print("\nBEST HYPERPARAMETERS — DWARF TYPE PRIORITIZED")
for k,v in best_params.items(): print(f"{k}: {v}")
print(f"Validation dwarf accuracy: {best_metrics['Dwarf_Acc']:.4f}")
print(f"Validation dwarf balanced accuracy: {best_metrics['Dwarf_BAcc']:.4f}")
print(f"Validation spectral-type accuracy: {best_metrics['SpT_Acc']:.4f}")
print(f"Validation spectral-type balanced accuracy: {best_metrics['SpT_BAcc']:.4f}")

print("\nTOP 10 CONFIGURATIONS")
print(tuning_df.head(10).to_string(index=False))

tuning_df.to_csv(FOLDER/"knn_hyperparameter_tuning.csv",index=False)
pd.DataFrame([{**best_params,"validation_dwarf_accuracy":best_metrics["Dwarf_Acc"],"validation_dwarf_balanced_accuracy":best_metrics["Dwarf_BAcc"],"validation_spt_accuracy":best_metrics["SpT_Acc"],"validation_spt_balanced_accuracy":best_metrics["SpT_BAcc"]}]).to_csv(FOLDER/"knn_best_hyperparameters.csv",index=False)
pd.DataFrame([ZHOU]).to_csv(FOLDER/"knn_reference_hyperparameters_zhou.csv",index=False)

model=Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler()),("knn",MultiOutputClassifier(KNeighborsClassifier(**best_params,n_jobs=-1),n_jobs=-1))])
model.fit(X_train,y_train)
print("\nFinal KNN trained.")

def plot_cm(y_true,y_pred,labels,names,file,kind):
    cm=confusion_matrix(y_true,y_pred,labels=labels,normalize="true")
    spt=kind=="spt"
    fig,ax=plt.subplots(figsize=(15,13) if spt else (10,8),dpi=100)
    im=ax.imshow(cm,cmap="Blues",vmin=0,vmax=1,aspect="auto")
    ax.set_xticks(range(len(names)),names,rotation=90 if spt else 45)
    ax.set_yticks(range(len(names)),names)
    ax.tick_params(axis="both",labelsize=22 if spt else 20)
    ax.set_xlabel("Predicted label",fontsize=29 if spt else 24,labelpad=16)
    ax.set_ylabel("Adopted label",fontsize=29 if spt else 24,labelpad=16)
    for i,j in np.ndindex(cm.shape):
        v=cm[i,j]; text="0" if v==0 else "<0.1" if v<.1 else f"{v:.2f}"
        ax.text(j,i,text,ha="center",va="center",fontsize=15 if spt else 24,color="white" if v>=.55 else "black")
    cbar=fig.colorbar(im,ax=ax,pad=.02 if spt else .03,shrink=.75 if spt else 1,fraction=.035,aspect=30)
    cbar.set_label("Normalized frequency",fontsize=27 if spt else 20,labelpad=12)
    cbar.ax.tick_params(labelsize=22 if spt else 14)
    plt.tight_layout()
    fig.savefig(file,dpi=600,bbox_inches="tight",facecolor="white")
    plt.show()
    return cm

def evaluate(name,d,X,y):
    true=y.to_numpy(int); pred=model.predict(X).astype(int)
    ts,td=true[:,0],true[:,1]; ps,pd_=pred[:,0],pred[:,1]; err=np.abs(ts-ps)
    summary={"Dataset":name,"Samples":len(y),"Features":X.shape[1],"Exact joint":np.mean((ts==ps)&(td==pd_)),"Joint ±1 SpT":np.mean((err<=1)&(td==pd_)),"Joint ±3 SpT":np.mean((err<=3)&(td==pd_)),"SpT accuracy":accuracy_score(ts,ps),"SpT balanced accuracy":balanced_accuracy_score(ts,ps),"SpT ±1":np.mean(err<=1),"SpT ±3":np.mean(err<=3),"Dwarf accuracy":accuracy_score(td,pd_),"Dwarf balanced accuracy":balanced_accuracy_score(td,pd_)}
    print("\n"+"="*72+f"\n{name}\n"+"="*72)
    for k,v in summary.items():
        if k not in ["Dataset","Samples","Features"]: print(f"{k:27s}: {v:.4f}")
    print("\nSPECTRAL-TYPE REPORT")
    print(classification_report(ts,ps,labels=SPT_LABELS,target_names=[spt_name(x) for x in SPT_LABELS],zero_division=0))
    print("\nDWARF-TYPE REPORT")
    print(classification_report(td,pd_,labels=DWARF_LABELS,target_names=[DWARF_NAMES[x] for x in DWARF_LABELS],zero_division=0))
    tag=name.lower()
    cm_spt=plot_cm(ts,ps,SPT_LABELS,[spt_name(x) for x in SPT_LABELS],FOLDER/f"cm_knn_spt_{tag}.png","spt")
    cm_dwarf=plot_cm(td,pd_,DWARF_LABELS,[DWARF_NAMES[x] for x in DWARF_LABELS],FOLDER/f"cm_knn_dwarf_{tag}.png","dwarf")
    ids=[c for c in ["Parent","Parent_Spectral_Type","Names","Type_Origen","Jmag","Hmag"] if c in d]
    out=d[ids].reset_index(drop=True).copy()
    out["true_spectral_type_int"],out["pred_spectral_type_int"]=ts,ps
    out["true_spectral_type"],out["pred_spectral_type"]=[spt_name(x) for x in ts],[spt_name(x) for x in ps]
    out["true_dwarf_type_int"],out["pred_dwarf_type_int"]=td,pd_
    out["true_dwarf_type"],out["pred_dwarf_type"]=[DWARF_NAMES.get(x,str(x)) for x in td],[DWARF_NAMES.get(x,str(x)) for x in pd_]
    out["spectral_type_error"],out["abs_spectral_type_error"]=ps-ts,err
    out["correct_spectral_type"],out["correct_spectral_type_pm1"],out["correct_spectral_type_pm3"]=ts==ps,err<=1,err<=3
    out["correct_dwarf_type"],out["correct_exact_match"]=td==pd_,(ts==ps)&(td==pd_)
    pd.DataFrame(cm_spt,index=[spt_name(x) for x in SPT_LABELS],columns=[spt_name(x) for x in SPT_LABELS]).to_csv(FOLDER/f"knn_cm_spt_{tag}.csv")
    pd.DataFrame(cm_dwarf,index=[DWARF_NAMES[x] for x in DWARF_LABELS],columns=[DWARF_NAMES[x] for x in DWARF_LABELS]).to_csv(FOLDER/f"knn_cm_dwarf_{tag}.csv")
    return out,summary

val_predictions,val_summary=evaluate("Validation",data["Validation"],X_val,y_val)
test_predictions,test_summary=evaluate("Test",data["Test"],X_test,y_test)
summary_df=pd.DataFrame([val_summary,test_summary])

print("\nSUMMARY")
print(summary_df.to_string(index=False))

val_predictions.to_csv(FOLDER/"knn_validation_predictions.csv",index=False)
test_predictions.to_csv(FOLDER/"knn_test_predictions.csv",index=False)
summary_df.to_csv(FOLDER/"knn_summary.csv",index=False)

print(f"\nAll results saved in: {FOLDER.resolve()}")

## K-Nearest Neighbors classification with fixed hyperparameters

Trains a K-Nearest Neighbors classifier using fixed hyperparameters, evaluates spectral type and peculiar class performance, and generates confusion matrices.

In [ ]:
from pathlib import Path
import re, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix

FOLDER=Path("KNN Results"); FOLDER.mkdir(exist_ok=True)
FILES={"Train":"train.csv","Validation":"val.csv","Test":"test.csv"}
TARGETS=["spectral_type_int","dwarf_type"]; MAGS=["Hmag","Jmag"]
FIELDREF=["fieldref_A","fieldref_YJ","fieldref_H","fieldref_K","fieldref_D"]
SPT_LABELS=list(range(30)); DWARF_LABELS=[-3,-2,-1,0,1,2]
DWARF_NAMES={-3:"esd",-2:"sd",-1:"d/sd",0:"field",1:"gamma",2:"beta"}
BIN_RE=re.compile(r"^\s*\(?\s*\d+(?:\.\d+)?\s*,\s*\d+(?:\.\d+)?\s*\)?\s*$")

plt.rcParams.update({"font.family":"serif","font.serif":["Times New Roman","Times","DejaVu Serif"],
                     "mathtext.fontset":"stix","font.size":13})

def spt_name(x):
    x=int(x); return f"M{x}" if x<10 else f"L{x-10}" if x<20 else f"T{x-20}" if x<30 else str(x)

def load(file):
    d=pd.read_csv(file,sep=None,engine="python"); d.columns=d.columns.astype(str).str.strip(); return d

data={k:load(v) for k,v in FILES.items()}

for name,d in data.items():
    missing=[c for c in TARGETS+MAGS+FIELDREF if c not in d]
    if missing: raise ValueError(f"{name} is missing: {missing}")

common=[c for c in data["Train"] if all(c in d for d in data.values())]
slopes=[c for c in common if c.startswith("slope_")]
bins=[c for c in common if BIN_RE.fullmatch(c)]
features=list(dict.fromkeys(MAGS+slopes+bins+FIELDREF))
features=[c for c in features if pd.to_numeric(data["Train"][c],errors="coerce").notna().any()]

if not slopes or not bins: raise ValueError("No slopes or spectral bins were detected.")

print(f"Magnitudes: {MAGS}\nSlopes: {len(slopes)}\nSpectral bins: {len(bins)}")
print(f"Field-reference features: {FIELDREF}\nTotal features: {len(features)}")
if len(slopes)!=16: print(f"WARNING: {len(slopes)} slopes detected; 16 expected.")
if len(bins)!=67: print(f"WARNING: {len(bins)} bins detected; 67 expected.")

def prepare(d):
    X=d[features].apply(pd.to_numeric,errors="coerce")
    y=d[TARGETS].apply(pd.to_numeric,errors="coerce")
    if y.isna().any().any(): raise ValueError("Missing or nonnumeric target values found.")
    return X,y.astype(int)

X_train,y_train=prepare(data["Train"])
X_val,y_val=prepare(data["Validation"])
X_test,y_test=prepare(data["Test"])
print(f"\nTrain: {X_train.shape} | Validation: {X_val.shape} | Test: {X_test.shape}")

# Exact Zhou et al. configuration
KNN_PARAMS={"n_neighbors":29,"metric":"minkowski","p":2,"weights":"uniform"}

model=Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler()),
    ("knn",MultiOutputClassifier(KNeighborsClassifier(**KNN_PARAMS,n_jobs=-1),n_jobs=1))
])

model.fit(X_train,y_train)
print("\nKNN trained with Zhou et al. hyperparameters:")
for k,v in KNN_PARAMS.items(): print(f"{k}: {v}")

def plot_cm(true,pred,labels,names,file,kind):
    cm=confusion_matrix(true,pred,labels=labels,normalize="true")
    spt=kind=="spt"
    fig,ax=plt.subplots(figsize=(15,13) if spt else (10,8),dpi=100)
    im=ax.imshow(cm,cmap="Blues",vmin=0,vmax=1,aspect="auto")
    ax.set_xticks(range(len(names)),names,rotation=90 if spt else 45)
    ax.set_yticks(range(len(names)),names)
    ax.tick_params(axis="both",labelsize=22 if spt else 20)
    ax.set_xlabel("Predicted label",fontsize=29 if spt else 24,labelpad=16)
    ax.set_ylabel("Adopted label",fontsize=29 if spt else 24,labelpad=16)

    for i,j in np.ndindex(cm.shape):
        v=cm[i,j]; text="0" if v==0 else "<0.01" if v<.01 else f"{v:.2f}"
        ax.text(j,i,text,ha="center",va="center",fontsize=15 if spt else 24,
                color="white" if v>=.55 else "black")

    cbar=fig.colorbar(im,ax=ax,pad=.02 if spt else .03,shrink=.75 if spt else 1,
                      fraction=.035,aspect=30)
    cbar.set_label("Normalized frequency",fontsize=27 if spt else 20,labelpad=12)
    cbar.ax.tick_params(labelsize=22 if spt else 14)
    plt.tight_layout()
    fig.savefig(file,dpi=600,bbox_inches="tight",facecolor="white")
    plt.show()
    return cm

def evaluate(name,d,X,y):
    true=y.to_numpy(int); pred=model.predict(X).astype(int)
    ts,td=true[:,0],true[:,1]; ps,pd_=pred[:,0],pred[:,1]; err=np.abs(ts-ps)

    summary={
        "Dataset":name,"Samples":len(y),"Features":X.shape[1],
        "Exact joint":np.mean((ts==ps)&(td==pd_)),
        "Joint ±1 SpT":np.mean((err<=1)&(td==pd_)),
        "Joint ±3 SpT":np.mean((err<=3)&(td==pd_)),
        "SpT accuracy":accuracy_score(ts,ps),
        "SpT balanced accuracy":balanced_accuracy_score(ts,ps),
        "SpT ±1":np.mean(err<=1),"SpT ±3":np.mean(err<=3),
        "Dwarf accuracy":accuracy_score(td,pd_),
        "Dwarf balanced accuracy":balanced_accuracy_score(td,pd_)
    }

    print("\n"+"="*72+f"\n{name}\n"+"="*72)
    for k,v in summary.items():
        if k not in ["Dataset","Samples","Features"]: print(f"{k:27s}: {v:.4f}")

    print("\nSPECTRAL-TYPE REPORT")
    print(classification_report(ts,ps,labels=SPT_LABELS,
          target_names=[spt_name(x) for x in SPT_LABELS],zero_division=0))

    print("\nDWARF-TYPE REPORT")
    print(classification_report(td,pd_,labels=DWARF_LABELS,
          target_names=[DWARF_NAMES[x] for x in DWARF_LABELS],zero_division=0))

    tag=name.lower()
    cm_spt=plot_cm(ts,ps,SPT_LABELS,[spt_name(x) for x in SPT_LABELS],
                   FOLDER/f"cm_knn_spt_{tag}.png","spt")
    cm_dwarf=plot_cm(td,pd_,DWARF_LABELS,[DWARF_NAMES[x] for x in DWARF_LABELS],
                     FOLDER/f"cm_knn_dwarf_{tag}.png","dwarf")

    ids=[c for c in ["Parent","Parent_Spectral_Type","Names","Type_Origen","Hmag","Jmag"] if c in d]
    out=d[ids].reset_index(drop=True).copy()
    out["true_spectral_type_int"],out["pred_spectral_type_int"]=ts,ps
    out["true_spectral_type"],out["pred_spectral_type"]=[spt_name(x) for x in ts],[spt_name(x) for x in ps]
    out["true_dwarf_type_int"],out["pred_dwarf_type_int"]=td,pd_
    out["true_dwarf_type"],out["pred_dwarf_type"]=[DWARF_NAMES.get(x,str(x)) for x in td],[DWARF_NAMES.get(x,str(x)) for x in pd_]
    out["spectral_type_error"],out["abs_spectral_type_error"]=ps-ts,err
    out["correct_spectral_type"],out["correct_spectral_type_pm1"],out["correct_spectral_type_pm3"]=ts==ps,err<=1,err<=3
    out["correct_dwarf_type"],out["correct_exact_match"]=td==pd_,(ts==ps)&(td==pd_)

    pd.DataFrame(cm_spt,index=[spt_name(x) for x in SPT_LABELS],
                 columns=[spt_name(x) for x in SPT_LABELS]).to_csv(FOLDER/f"knn_cm_spt_{tag}.csv")
    pd.DataFrame(cm_dwarf,index=[DWARF_NAMES[x] for x in DWARF_LABELS],
                 columns=[DWARF_NAMES[x] for x in DWARF_LABELS]).to_csv(FOLDER/f"knn_cm_dwarf_{tag}.csv")
    return out,summary

val_predictions,val_summary=evaluate("Validation",data["Validation"],X_val,y_val)
test_predictions,test_summary=evaluate("Test",data["Test"],X_test,y_test)
summary=pd.DataFrame([val_summary,test_summary])

print("\nSUMMARY")
print(summary.to_string(index=False))

val_predictions.to_csv(FOLDER/"knn_validation_predictions.csv",index=False)
test_predictions.to_csv(FOLDER/"knn_test_predictions.csv",index=False)
summary.to_csv(FOLDER/"knn_summary.csv",index=False)
pd.DataFrame([KNN_PARAMS]).to_csv(FOLDER/"knn_hyperparameters_zhou.csv",index=False)

print(f"\nAll KNN results saved in: {FOLDER.resolve()}")

# Convolutional Neural Network

In [ ]:
from pathlib import Path
import re, random, numpy as np, pandas as pd, matplotlib.pyplot as plt, tensorflow as tf
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix

FOLDER=Path("/content/CNN Results - 30"); FOLDER.mkdir(parents=True,exist_ok=True)
FILES={"Train":"/content/train.csv","Validation":"/content/val.csv","Test":"/content/test.csv"}
TARGETS=["spectral_type_int","dwarf_type"]; MAGS=["Jmag","Hmag"]
FIELDREF=["fieldref_A","fieldref_YJ","fieldref_H","fieldref_K","fieldref_D"]
SPT_LABELS=list(range(30)); DWARF_LABELS=[-3,-2,-1,0,1,2]
DWARF_NAMES={-3:"esd",-2:"sd",-1:"d/sd",0:"field",1:"gamma",2:"beta"}
DWARF_TO_IDX={v:i for i,v in enumerate(DWARF_LABELS)}
IDX_TO_DWARF={i:v for v,i in DWARF_TO_IDX.items()}
BIN_RE=re.compile(r"^\s*\(?\s*\d+(?:\.\d+)?\s*,\s*\d+(?:\.\d+)?\s*\)?\s*$")
N_TRIALS,SEED=30,42

BASE_CONFIG={"filters1":32,"filters2":64,"filters3":96,"kernel1":5,"kernel2":5,"kernel3":3,"drop1":0.20,"drop2":0.25,"drop3":0.35,"dense":64,"drop_dense":0.40,"l2":1e-4,"lr":5e-4,"batch_size":32}
SPACE={"filters1":[24,32,48],"filters2":[48,64,96],"filters3":[64,96,128],"kernel1":[3,5,7],"kernel2":[3,5],"kernel3":[3,5],"drop1":[0.15,0.20,0.25],"drop2":[0.20,0.25,0.30],"drop3":[0.30,0.35,0.40],"dense":[48,64,96,128],"drop_dense":[0.30,0.40,0.50],"l2":[5e-5,1e-4,2e-4],"lr":[1e-4,3e-4,5e-4,1e-3],"batch_size":[32,64]}

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
plt.rcParams.update({"font.family":"serif","font.serif":["Times New Roman","Times","DejaVu Serif"],"mathtext.fontset":"stix","font.size":13})

def spt_name(x):
    x=int(x); return f"M{x}" if x<10 else f"L{x-10}" if x<20 else f"T{x-20}" if x<30 else str(x)

def load_csv(file):
    d=pd.read_csv(file,sep=None,engine="python"); d.columns=d.columns.astype(str).str.strip(); return d

data={name:load_csv(file) for name,file in FILES.items()}

for name,d in data.items():
    missing=[c for c in TARGETS+MAGS+FIELDREF if c not in d]
    if missing: raise ValueError(f"{name} is missing: {missing}")

common=[c for c in data["Train"] if all(c in d for d in data.values())]
slope_cols=[c for c in common if c.startswith("slope_")]
bin_cols=sorted([c for c in common if BIN_RE.fullmatch(c)], key=lambda x: float(re.findall(r"\d+(?:\.\d+)?",x)[0]))
feature_cols=list(dict.fromkeys(MAGS+slope_cols+bin_cols+FIELDREF))
feature_cols=[c for c in feature_cols if pd.to_numeric(data["Train"][c],errors="coerce").notna().any()]
if not slope_cols or not bin_cols: raise ValueError("No slopes or spectral bins were detected.")

print(f"Magnitudes: {MAGS}\nSlopes: {len(slope_cols)}\nSpectral bins: {len(bin_cols)}")
print(f"Field-reference features: {FIELDREF}\nTotal features: {len(feature_cols)}")
if len(slope_cols)!=16: print(f"WARNING: {len(slope_cols)} slopes detected; 16 expected.")
if len(bin_cols)!=67: print(f"WARNING: {len(bin_cols)} bins detected; 67 expected.")

def prepare(d):
    X=d[feature_cols].apply(pd.to_numeric,errors="coerce")
    y=d[TARGETS].apply(pd.to_numeric,errors="coerce")
    if y.isna().any().any(): raise ValueError("Missing or nonnumeric target values found.")
    y_spt=y["spectral_type_int"].astype(int).to_numpy()
    y_dwarf=y["dwarf_type"].astype(int).map(DWARF_TO_IDX).to_numpy()
    return X,y.astype(int),y_spt,y_dwarf

X_train,y_train_df,y_train_spt,y_train_dwarf=prepare(data["Train"])
X_val,y_val_df,y_val_spt,y_val_dwarf=prepare(data["Validation"])
X_test,y_test_df,y_test_spt,y_test_dwarf=prepare(data["Test"])
print(f"\nTrain: {X_train.shape} | Validation: {X_val.shape} | Test: {X_test.shape}")

imputer,scaler=SimpleImputer(strategy="median"),StandardScaler()
Xt=imputer.fit_transform(X_train); Xv=imputer.transform(X_val); Xte=imputer.transform(X_test)
Xt=scaler.fit_transform(Xt); Xv=scaler.transform(Xv); Xte=scaler.transform(Xte)
Xt,Xv,Xte=Xt[...,None],Xv[...,None],Xte[...,None]

def sample_config():
    return {k:random.choice(v) for k,v in SPACE.items()}

def build_model(cfg,n_features):
    reg=tf.keras.regularizers.l2(cfg["l2"])
    inp=tf.keras.Input(shape=(n_features,1),name="input")
    x=tf.keras.layers.Conv1D(cfg["filters1"],cfg["kernel1"],padding="same",activation="relu",kernel_regularizer=reg)(inp)
    x=tf.keras.layers.BatchNormalization()(x); x=tf.keras.layers.MaxPooling1D(2)(x); x=tf.keras.layers.Dropout(cfg["drop1"])(x)
    x=tf.keras.layers.Conv1D(cfg["filters2"],cfg["kernel2"],padding="same",activation="relu",kernel_regularizer=reg)(x)
    x=tf.keras.layers.BatchNormalization()(x); x=tf.keras.layers.MaxPooling1D(2)(x); x=tf.keras.layers.Dropout(cfg["drop2"])(x)
    x=tf.keras.layers.Conv1D(cfg["filters3"],cfg["kernel3"],padding="same",activation="relu",kernel_regularizer=reg)(x)
    x=tf.keras.layers.BatchNormalization()(x); x=tf.keras.layers.Dropout(cfg["drop3"])(x)
    x=tf.keras.layers.GlobalAveragePooling1D()(x)
    x=tf.keras.layers.Dense(cfg["dense"],activation="relu",kernel_regularizer=reg)(x)
    x=tf.keras.layers.Dropout(cfg["drop_dense"])(x)
    spt_out=tf.keras.layers.Dense(30,activation="softmax",name="spt")(x)
    dwarf_out=tf.keras.layers.Dense(6,activation="softmax",name="dwarf")(x)
    model=tf.keras.Model(inp,[spt_out,dwarf_out])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=cfg["lr"]),
                  loss={"spt":"sparse_categorical_crossentropy","dwarf":"sparse_categorical_crossentropy"},
                  metrics={"spt":["accuracy"],"dwarf":["accuracy"]})
    return model

def predict_labels(model,X):
    ps,pg=model.predict(X,verbose=0)
    return np.argmax(ps,axis=1),np.array([IDX_TO_DWARF[i] for i in np.argmax(pg,axis=1)])

callbacks=[
    tf.keras.callbacks.EarlyStopping(monitor="val_loss",patience=12,min_delta=1e-4,restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss",factor=0.5,patience=5,min_lr=1e-6)
]

configs=[("Reference",BASE_CONFIG.copy())]
seen={tuple(sorted(BASE_CONFIG.items()))}
while len(configs)<N_TRIALS+1:
    c=sample_config(); key=tuple(sorted(c.items()))
    if key not in seen: seen.add(key); configs.append(("Random",c))

results,best_rank,best_config,best_model,best_metrics=[],(-np.inf,)*4,None,None,None

for i,(source,cfg) in enumerate(configs,1):
    tf.keras.backend.clear_session(); tf.random.set_seed(SEED+i)
    model=build_model(cfg,len(feature_cols))
    hist=model.fit(Xt,{"spt":y_train_spt,"dwarf":y_train_dwarf},validation_data=(Xv,{"spt":y_val_spt,"dwarf":y_val_dwarf}),
                   epochs=300,batch_size=cfg["batch_size"],callbacks=callbacks,verbose=0)
    ps,pd_=predict_labels(model,Xv)
    spt_acc=accuracy_score(y_val_spt,ps); spt_bacc=balanced_accuracy_score(y_val_spt,ps)
    dwarf_true=y_val_df["dwarf_type"].astype(int).to_numpy()
    dwarf_acc=accuracy_score(dwarf_true,pd_); dwarf_bacc=balanced_accuracy_score(dwarf_true,pd_)
    row={"source":source,**cfg,"epochs_trained":len(hist.history["loss"]),"best_val_loss":min(hist.history["val_loss"]),"SpT_Acc":spt_acc,"SpT_BAcc":spt_bacc,"Dwarf_Acc":dwarf_acc,"Dwarf_BAcc":dwarf_bacc,"Mean_Acc":(spt_acc+dwarf_acc)/2,"Mean_BAcc":(spt_bacc+dwarf_bacc)/2}
    results.append(row)
    rank=(dwarf_bacc,dwarf_acc,spt_bacc,spt_acc)
    if rank>best_rank: best_rank,best_config,best_model,best_metrics=rank,cfg.copy(),model,row.copy()
    print(f"{i:02d} | {source:9s} | epochs={row['epochs_trained']:3d} | val_loss={row['best_val_loss']:.4f} | SpT Acc={spt_acc:.4f} BAcc={spt_bacc:.4f} | Dwarf Acc={dwarf_acc:.4f} BAcc={dwarf_bacc:.4f}")

tuning_df=pd.DataFrame(results).sort_values(["Dwarf_BAcc","Dwarf_Acc","SpT_BAcc","SpT_Acc"],ascending=False).reset_index(drop=True)
print("\nBEST CNN CONFIGURATION — DWARF TYPE PRIORITIZED")
for k,v in best_config.items(): print(f"{k}: {v}")
print(f"Validation dwarf accuracy: {best_metrics['Dwarf_Acc']:.4f}")
print(f"Validation dwarf balanced accuracy: {best_metrics['Dwarf_BAcc']:.4f}")
print(f"Validation spectral-type accuracy: {best_metrics['SpT_Acc']:.4f}")
print(f"Validation spectral-type balanced accuracy: {best_metrics['SpT_BAcc']:.4f}")
print("\nTOP 10 CONFIGURATIONS")
print(tuning_df.head(10).to_string(index=False))

tuning_df.to_csv(FOLDER/"cnn_hyperparameter_tuning.csv",index=False)
pd.DataFrame([{**best_config,"validation_dwarf_accuracy":best_metrics["Dwarf_Acc"],"validation_dwarf_balanced_accuracy":best_metrics["Dwarf_BAcc"],"validation_spt_accuracy":best_metrics["SpT_Acc"],"validation_spt_balanced_accuracy":best_metrics["SpT_BAcc"]}]).to_csv(FOLDER/"cnn_best_hyperparameters.csv",index=False)
pd.DataFrame([BASE_CONFIG]).to_csv(FOLDER/"cnn_reference_architecture.csv",index=False)

best_model.save(FOLDER/"cnn_best_model.keras")
model=best_model
print("\nBest CNN selected from validation results.")

def plot_cm(y_true,y_pred,labels,names,file,kind):
    cm=confusion_matrix(y_true,y_pred,labels=labels,normalize="true")
    spt=kind=="spt"
    fig,ax=plt.subplots(figsize=(15,13) if spt else (10,8),dpi=100)
    im=ax.imshow(cm,cmap="Blues",vmin=0,vmax=1,aspect="auto")
    ax.set_xticks(range(len(names)),names,rotation=90 if spt else 45)
    ax.set_yticks(range(len(names)),names)
    ax.tick_params(axis="both",labelsize=22 if spt else 20)
    ax.set_xlabel("Predicted label",fontsize=29 if spt else 24,labelpad=16)
    ax.set_ylabel("Adopted label",fontsize=29 if spt else 24,labelpad=16)
    for i,j in np.ndindex(cm.shape):
        v=cm[i,j]; text="0" if v==0 else "<0.1" if v<.1 else f"{v:.2f}"
        ax.text(j,i,text,ha="center",va="center",fontsize=15 if spt else 24,color="white" if v>=.55 else "black")
    cbar=fig.colorbar(im,ax=ax,pad=.02 if spt else .03,shrink=.75 if spt else 1,fraction=.035,aspect=30)
    cbar.set_label("Normalized frequency",fontsize=27 if spt else 20,labelpad=12)
    cbar.ax.tick_params(labelsize=22 if spt else 14)
    plt.tight_layout()
    fig.savefig(file,dpi=600,bbox_inches="tight",facecolor="white")
    plt.show()
    return cm

def evaluate(name,d,X,y_df,y_spt,y_dwarf_idx):
    ps,pd_=predict_labels(model,X)
    ts=y_df["spectral_type_int"].astype(int).to_numpy()
    td=y_df["dwarf_type"].astype(int).to_numpy()
    err=np.abs(ts-ps)
    summary={"Dataset":name,"Samples":len(y_df),"Features":X.shape[1],"Exact joint":np.mean((ts==ps)&(td==pd_)),"Joint ±1 SpT":np.mean((err<=1)&(td==pd_)),"Joint ±3 SpT":np.mean((err<=3)&(td==pd_)),"SpT accuracy":accuracy_score(ts,ps),"SpT balanced accuracy":balanced_accuracy_score(ts,ps),"SpT ±1":np.mean(err<=1),"SpT ±3":np.mean(err<=3),"Dwarf accuracy":accuracy_score(td,pd_),"Dwarf balanced accuracy":balanced_accuracy_score(td,pd_)}
    print("\n"+"="*72+f"\n{name}\n"+"="*72)
    for k,v in summary.items():
        if k not in ["Dataset","Samples","Features"]: print(f"{k:27s}: {v:.4f}")
    print("\nSPECTRAL-TYPE REPORT")
    print(classification_report(ts,ps,labels=SPT_LABELS,target_names=[spt_name(x) for x in SPT_LABELS],zero_division=0))
    print("\nDWARF-TYPE REPORT")
    print(classification_report(td,pd_,labels=DWARF_LABELS,target_names=[DWARF_NAMES[x] for x in DWARF_LABELS],zero_division=0))
    tag=name.lower()
    cm_spt=plot_cm(ts,ps,SPT_LABELS,[spt_name(x) for x in SPT_LABELS],FOLDER/f"cm_cnn_spt_{tag}.png","spt")
    cm_dwarf=plot_cm(td,pd_,DWARF_LABELS,[DWARF_NAMES[x] for x in DWARF_LABELS],FOLDER/f"cm_cnn_dwarf_{tag}.png","dwarf")
    ids=[c for c in ["Parent","Parent_Spectral_Type","Names","Type_Origen","Jmag","Hmag"] if c in d]
    out=d[ids].reset_index(drop=True).copy()
    out["true_spectral_type_int"],out["pred_spectral_type_int"]=ts,ps
    out["true_spectral_type"],out["pred_spectral_type"]=[spt_name(x) for x in ts],[spt_name(x) for x in ps]
    out["true_dwarf_type_int"],out["pred_dwarf_type_int"]=td,pd_
    out["true_dwarf_type"],out["pred_dwarf_type"]=[DWARF_NAMES.get(x,str(x)) for x in td],[DWARF_NAMES.get(x,str(x)) for x in pd_]
    out["spectral_type_error"],out["abs_spectral_type_error"]=ps-ts,err
    out["correct_spectral_type"],out["correct_spectral_type_pm1"],out["correct_spectral_type_pm3"]=ts==ps,err<=1,err<=3
    out["correct_dwarf_type"],out["correct_exact_match"]=td==pd_,(ts==ps)&(td==pd_)
    pd.DataFrame(cm_spt,index=[spt_name(x) for x in SPT_LABELS],columns=[spt_name(x) for x in SPT_LABELS]).to_csv(FOLDER/f"cnn_cm_spt_{tag}.csv")
    pd.DataFrame(cm_dwarf,index=[DWARF_NAMES[x] for x in DWARF_LABELS],columns=[DWARF_NAMES[x] for x in DWARF_LABELS]).to_csv(FOLDER/f"cnn_cm_dwarf_{tag}.csv")
    return out,summary

val_predictions,val_summary=evaluate("Validation",data["Validation"],Xv,y_val_df,y_val_spt,y_val_dwarf)
test_predictions,test_summary=evaluate("Test",data["Test"],Xte,y_test_df,y_test_spt,y_test_dwarf)
summary_df=pd.DataFrame([val_summary,test_summary])

print("\nSUMMARY")
print(summary_df.to_string(index=False))

val_predictions.to_csv(FOLDER/"cnn_validation_predictions.csv",index=False)
test_predictions.to_csv(FOLDER/"cnn_test_predictions.csv",index=False)
summary_df.to_csv(FOLDER/"cnn_summary.csv",index=False)

print(f"\nAll results saved in: {FOLDER.resolve()}")